Dataset Final

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import GroupShuffleSplit

SEED = 42
df = pd.read_csv("/content/Final Data.csv", low_memory=False)
is_atk = df["is_attack"].astype(str).str.lower().eq("true")

inf = np.isinf(df.select_dtypes("number")).sum()
print(df.shape, "| benign", (~is_atk).sum(), "| attack", is_atk.sum())
print("inf:", inf[inf > 0].to_dict() or "none")

In [ ]:
benign = df[~is_atk]                       # all 598 -> prompts

tr, te = [], []
for cls in [False, True]:                  # 70/30, same ratio, grouped by flow_key
    part = df[is_atk == cls]
    a, b = next(GroupShuffleSplit(1, test_size=0.30, random_state=SEED).split(part, groups=part["flow_key"]))
    tr.append(part.iloc[a]); te.append(part.iloc[b])
train = pd.concat(tr).sample(frac=1, random_state=SEED)
test  = pd.concat(te).sample(frac=1, random_state=SEED)
benign_train = train[~train["is_attack"].astype(str).str.lower().eq("true")]   # ~404 -> TSTR training

assert not set(train.flow_key) & set(test.flow_key)
for name, d in [("benign", benign), ("benign_train", benign_train), ("train", train), ("test", test)]:
    v = d["is_attack"].astype(str).str.lower().eq("true")
    print(f"{name:12s} {len(d):5d} rows | benign {(~v).sum():4d} | attack {v.sum():4d} | attack% {v.mean():.1%}")

In [ ]:
from google.colab import files
out = {"benign.csv": benign, "benign_train.csv": benign_train, "train.csv": train, "test.csv": test}
for f, d in out.items():
    d.to_csv(f, index=False); files.download(f)

Prompt Generation Part

In [ ]:
import json, re, numpy as np, pandas as pd
from pathlib import Path

OUT = Path("/content/rules_out")
OUT.mkdir(exist_ok=True, parents=True)
df = pd.read_csv("/content/benign.csv", low_memory=False)
print("benign:", df.shape)

TEXT = df.select_dtypes(exclude="number").columns.tolist()
NUM = df.select_dtypes("number").columns.tolist()
nuniq = df[NUM].nunique(dropna=True)
ALL_NA = [c for c in NUM if df[c].isna().all()]
CONST = {}
for c in NUM:
    if nuniq[c] == 1 and c not in ALL_NA:
        CONST[c] = float(df[c].dropna().iloc[0])
VARY = [c for c in NUM if nuniq[c] > 1]

print()
print("text cols:", len(TEXT), TEXT)
print("always-empty cols:", len(ALL_NA), ALL_NA)
print("constant cols:", len(CONST))
print(pd.Series(CONST).to_string())
print()
print("varying numeric cols:", len(VARY))

In [ ]:
#Empty Value Rules
na_cols = [c for c in df.columns if df[c].isna().any() and c not in ALL_NA]

groups = {}
for c in na_cols:
    key = tuple(df[c].isna().values)
    groups.setdefault(key, []).append(c)

INT_NA = df["mean_hop_latency"].isna()
CONDS = {
    "duration == 0": df["duration"] == 0,
    "pkts_fwd == 1": df["pkts_fwd"] == 1,
    "pkts_fwd <= 2": df["pkts_fwd"] <= 2,
    "pkts_fwd == 1 OR INT missing": (df["pkts_fwd"] == 1) | INT_NA,
}

EMPTY_RULES = []
for key, cols in groups.items():
    m = pd.Series(np.array(key), index=df.index)
    exact, must, only = [], [], []
    for name, cond in CONDS.items():
        if (m == cond).all(): exact.append(name)
        if (~cond | m).all(): must.append(name)
        if (~m | cond).all(): only.append(name)
    EMPTY_RULES.append(dict(columns=cols, n_empty=int(m.sum()),
                            exact_if=exact, must_be_empty_if=must,
                            can_be_empty_only_if=only))

print("ALWAYS EMPTY:", ALL_NA)
for r in EMPTY_RULES:
    print()
    print(f"{len(r['columns'])} cols | empty in {r['n_empty']} rows")
    print("  cols:", r["columns"])
    print("  EXACT (empty <=> cond):", r["exact_if"] or "-")
    print("  MUST be empty if     :", r["must_be_empty_if"] or "-")
    print("  CAN be empty only if :", r["can_be_empty_only_if"] or "-")

In [ ]:
#Formulas
FLAGS = ["syn", "ack", "fin", "rst", "psh", "urg"]
NAMED = [
    ("pkts_total", "pkts_fwd + pkts_bwd"),
    ("bytes_total", "bytes_fwd + bytes_bwd"),
    ("mean_pkt_size", "bytes_total / pkts_total"),
    ("bytes_per_pkt_total", "bytes_total / pkts_total"),
    ("bytes_per_pkt_fwd", "bytes_fwd / pkts_fwd"),
    ("pps_fwd", "pkts_fwd / duration"),
    ("pps_bwd", "pkts_bwd / duration"),
    ("pps_total", "pkts_total / duration"),
    ("bps_fwd", "bytes_fwd / duration"),
    ("bps_bwd", "bytes_bwd / duration"),
    ("bps_total", "bytes_total / duration"),
    ("fwd_only_duration", "duration"),
    ("fwd_share_pkts", "pkts_fwd / pkts_total"),
    ("fwd_share_bytes", "bytes_fwd / bytes_total"),
    ("reverse_pkt_fraction", "pkts_bwd / pkts_total"),
    ("reverse_byte_fraction", "bytes_bwd / bytes_total"),
    ("pkts_bwd_zero", "1.0 * (pkts_bwd == 0)"),
    ("bytes_bwd_zero", "1.0 * (bytes_bwd == 0)"),
    ("has_response", "1.0 * (pkts_bwd > 0)"),
    ("is_fwd_only", "1.0 * ((pkts_fwd > 0) & (pkts_bwd == 0))"),
    ("is_bwd_only", "1.0 * ((pkts_bwd > 0) & (pkts_fwd == 0))"),
    ("syn_fraction_fwd", "syn_count_fwd / pkts_fwd"),
    ("syn_ack_gap", "syn_count_fwd - syn_count_bwd"),
    ("flow_share_of_target_service_pkts", "pkts_total / target_service_pkts_total"),
    ("flow_share_of_target_service_bytes", "bytes_total / target_service_bytes_total"),
    ("flow_share_of_target_ip_pkts", "pkts_total / target_ip_pkts_total"),
    ("flow_share_of_target_ip_bytes", "bytes_total / target_ip_bytes_total"),
]
for f in FLAGS:
    NAMED += [(f"{f}_ratio_fwd", f"{f}_count_fwd / pkts_fwd"),
              (f"{f}_ratio_bwd", f"{f}_count_bwd / pkts_bwd"),
              (f"{f}_fraction", f"({f}_count_fwd + {f}_count_bwd) / pkts_total"),
              (f"{f}_no_reverse", f"1.0 * (({f}_count_fwd > 0) & (pkts_bwd == 0))")]

def calc(expr, frame):
    env = {c: frame[c].astype(float) for c in NUM}
    with np.errstate(all="ignore"):
        v = eval(expr, {}, env)
    if not isinstance(v, pd.Series): v = pd.Series(v, index=frame.index)
    return v.replace([np.inf, -np.inf], np.nan)

def holds(target, expr):
    y, e = df[target].astype(float), calc(expr, df)
    same = np.isclose(y, e, rtol=1e-4, atol=1e-9) | (y.isna() & e.isna())
    return float(same.mean())

DERIVED, REJECTED = {}, []
for t, expr in NAMED:
    if t not in df.columns: continue
    try: s = holds(t, expr)
    except KeyError: continue
    if s == 1.0: DERIVED[t] = expr
    else: REJECTED.append((t, expr, round(s, 4)))

def unit(c):
    if re.search(r"(^is_|^has_|_zero$|no_reverse|syn_ack_gap)", c): return None
    if "duration" in c: return (0, 0, 1)
    if "pps" in c: return (1, 0, -1)
    if "bps" in c: return (0, 1, -1)
    if re.search(r"(bytes_per_pkt|pkt_size)", c): return (-1, 1, 0)
    if re.search(r"(ratio|fraction|share)", c): return (0, 0, 0)
    if "bytes" in c: return (0, 1, 0)
    if re.search(r"(pkts|count)", c): return (1, 0, 0)
    return None

ID_NUM = ["timeframe_id", "proto", "port_a", "port_b"]
BASE = [c for c in VARY if c not in DERIVED and c not in ID_NUM and unit(c)]
U = {c: unit(c) for c in BASE}
X = {c: df[c].astype(float).values for c in BASE}
EXTRA = []
with np.errstate(all="ignore"):
    for a in BASE:
        for b in BASE:
            if a == b: continue
            ops = [("/", X[a] / X[b], tuple(i - j for i, j in zip(U[a], U[b])))]
            if U[a] == U[b]:
                ops.append(("-", X[a] - X[b], U[a]))
            if a < b:
                ops.append(("*", X[a] * X[b], tuple(i + j for i, j in zip(U[a], U[b]))))
                if U[a] == U[b]: ops.append(("+", X[a] + X[b], U[a]))
            for op, e, u in ops:
                for c in BASE:
                    if c in (a, b) or U[c] != u: continue
                    y = X[c]
                    m = ~np.isnan(y) & np.isfinite(e) & (y != 0)
                    if m.sum() < 20: continue
                    if np.allclose(y[m], e[m], rtol=1e-4) and holds(c, f"{a} {op} {b}") == 1:
                        EXTRA.append((c, f"{a} {op} {b}", int(m.sum())))

COPIES = []
for c in [x for x in VARY if x not in DERIVED and x not in ID_NUM]:
    for a in [x for x in VARY if x != c and x not in ID_NUM and x not in DERIVED]:
        y, e = df[c].astype(float), df[a].astype(float)
        m = y.notna() & e.notna() & (y != 0)
        if m.sum() >= 20 and holds(c, a) == 1:
            COPIES.append((c, a, int(m.sum())))

FLOWKEY_OK = float((df.flow_key == (
    df.timeframe_id.astype(str) + "|" + df.proto.astype(str) + "|" +
    df.ip_a + "|" + df.ip_b + "|" +
    df.port_a.astype(str) + "|" + df.port_b.astype(str))).mean())

print(f"(A) named formulas verified 100%: {len(DERIVED)}")
for t, e in DERIVED.items(): print(f"   {t:36s} = {e}")
print()
print("(A) named formulas REJECTED (not 100%):")
for t, e, s in REJECTED: print(f"   {t:36s} = {e}   [{s:.1%}]")
print()
print(f"(B) extra formulas found by brute force (+ - * /): {len(EXTRA)}")
for t, e, n in EXTRA: print(f"   {t:36s} = {e}   (evidence {n})")
print(f"(C) copies found (C = A): {COPIES}")
print()
print(f"flow_key = timeframe|proto|ip_a|ip_b|port_a|port_b -> {FLOWKEY_OK:.0%}")

In [ ]:
rebuilt = df.copy()
for t in DERIVED: rebuilt[t] = np.nan
for t, expr in DERIVED.items(): rebuilt[t] = calc(expr, rebuilt)

bad = []
for t in DERIVED:
    y, e = df[t].astype(float), rebuilt[t]
    same = np.isclose(y, e, rtol=1e-4, atol=1e-9) | (y.isna() & e.isna())
    if same.mean() < 1: bad.append((t, round(float(same.mean()), 4)))
print(f"rebuild check on {len(DERIVED)} derived cols:",
      "ALL match 100%" if not bad else bad)

In [ ]:
# CAPTURE_PAT = re.compile(r"_bwd|bwd_|truncated_ip")
# PINNED = {c: v for c, v in CONST.items()
#           if CAPTURE_PAT.search(c) and c not in DERIVED}
# PINNED["ip_b"] = df["ip_b"].iloc[0]
# FREE_CONST = [c for c in CONST if c not in PINNED and c not in DERIVED]

# LABEL = ["is_attack"]
# SKIP = set(DERIVED) | set(PINNED) | set(ALL_NA) | {"flow_key"} | set(LABEL)
# CORE = [c for c in df.columns if c not in SKIP]

# DTYPES = {}
# for c in df.columns:
#     if pd.api.types.is_integer_dtype(df[c]): DTYPES[c] = "int"
#     elif pd.api.types.is_float_dtype(df[c]): DTYPES[c] = "float"
#     else: DTYPES[c] = "text"

# CONSTRAINTS = []
# for c in CORE:
#     if DTYPES[c] == "text": continue
#     s = df[c].dropna()
#     if (s >= 0).all(): CONSTRAINTS.append(f"{c} >= 0")
#     if re.search(r"(ratio|fraction|share)", c) and s.between(0, 1).all():
#         CONSTRAINTS.append(f"0 <= {c} <= 1")
# for lo, mid, hi in [("pkt_size_min", "mean_pkt_size", "pkt_size_max"),
#                     ("iat_min_int_ns", "mean_iat_int_ns", "iat_max_int_ns"),
#                     ("latency_min", "mean_hop_latency", "latency_max")]:
#     s = (df[lo] <= df[mid] + 1e-9) & (df[mid] <= df[hi] + 1e-9)
#     if s[df[mid].notna()].all(): CONSTRAINTS.append(f"{lo} <= {mid} <= {hi}")
# for f in ["syn", "ack", "fin", "rst", "psh", "urg"]:
#     if (df[f"{f}_count_fwd"] <= df["pkts_fwd"]).all():
#         CONSTRAINTS.append(f"{f}_count_fwd <= pkts_fwd")

# STATS = {}
# for c in CORE:
#     if DTYPES[c] == "text": continue
#     s = df[c]
#     qs = s.quantile([.05, .25, .5, .75, .95])
#     STATS[c] = {"dtype": DTYPES[c],
#                 "p5": qs.iloc[0], "p25": qs.iloc[1], "p50": qs.iloc[2],
#                 "p75": qs.iloc[3], "p95": qs.iloc[4],
#                 "min": s.min(), "max": s.max(),
#                 "zero_rate": round(float((s == 0).mean()), 3),
#                 "empty_rate": round(float(s.isna().mean()), 3)}

# IDS = {"flow_key": "timeframe_id|proto|ip_a|ip_b|port_a|port_b",
#        "ip_a_format": "anon_ip_N",
#        "ip_a_unique": int(df.ip_a.nunique()),
#        "ip_b": PINNED["ip_b"],
#        "port_a_top": df.port_a.value_counts().head(5).to_dict(),
#        "port_b_range": [int(df.port_b.min()), int(df.port_b.max())],
#        "timeframe_range": [int(df.timeframe_id.min()), int(df.timeframe_id.max())],
#        "proto_values": df.proto.value_counts().to_dict()}

# print("pinned (capture):", len(PINNED), list(PINNED))
# print()
# print("constant in benign but LLM-free:", FREE_CONST)
# print()
# print("CORE columns the LLM generates:", len(CORE))
# print(CORE)
# print()
# print("constraints:", len(CONSTRAINTS))
# print()
# print("ID formats:")
# print(json.dumps(IDS, default=str, indent=1))

In [ ]:
CAPTURE_PAT = re.compile(r"_bwd|bwd_|truncated_ip")
PINNED = {c: v for c, v in CONST.items()
          if CAPTURE_PAT.search(c) and c not in DERIVED}
PINNED["ip_b"] = df["ip_b"].iloc[0]
FREE_CONST = [c for c in CONST if c not in PINNED and c not in DERIVED]

LABEL = ["is_attack"]
SKIP = set(DERIVED) | set(PINNED) | set(ALL_NA) | {"flow_key"} | set(LABEL)
CORE = [c for c in df.columns if c not in SKIP]

DTYPES = {}
for c in df.columns:
    if pd.api.types.is_integer_dtype(df[c]): DTYPES[c] = "int"
    elif pd.api.types.is_float_dtype(df[c]): DTYPES[c] = "float"
    else: DTYPES[c] = "text"

CONSTRAINTS = []
for c in CORE:
    if DTYPES[c] == "text": continue
    s = df[c].dropna()
    if (s >= 0).all(): CONSTRAINTS.append(f"{c} >= 0")
    if re.search(r"(ratio|fraction|share)", c) and s.between(0, 1).all():
        CONSTRAINTS.append(f"0 <= {c} <= 1")
for lo, mid, hi in [("pkt_size_min", "mean_pkt_size", "pkt_size_max"),
                    ("iat_min_int_ns", "mean_iat_int_ns", "iat_max_int_ns"),
                    ("latency_min", "mean_hop_latency", "latency_max")]:
    s = (df[lo] <= df[mid] + 1e-9) & (df[mid] <= df[hi] + 1e-9)
    if s[df[mid].notna()].all(): CONSTRAINTS.append(f"{lo} <= {mid} <= {hi}")
for f in ["syn", "ack", "fin", "rst", "psh", "urg"]:
    if (df[f"{f}_count_fwd"] <= df["pkts_fwd"]).all():
        CONSTRAINTS.append(f"{f}_count_fwd <= pkts_fwd")

# quiet-window fix: target_* stats only from attack-free timeframes
# schedule from labeling schema: active 5-10, decay 11
ATTACK_TF = list(range(5, 12))
QUIET = ~df["timeframe_id"].isin(ATTACK_TF)
CTX_TARGET = [c for c in CORE if c.startswith("target_")]
print("quiet-window benign rows:", int(QUIET.sum()), "| used for:", CTX_TARGET)

CONSTRAINTS += [
    "mean_hop_count = 1 whenever INT data is present",
    "source_flow_count = number of flows from the same ip_a in the same timeframe_id",
    "source_pkts_total = sum of pkts_total over those flows",
    "source_bytes_total = sum of bytes_total over those flows",
    "pkts_fwd == 1  <=>  duration == 0",
]

STATS = {}
for c in CORE:
    if DTYPES[c] == "text": continue
    s = df.loc[QUIET, c] if c in CTX_TARGET else df[c]
    qs = s.quantile([.05, .25, .5, .75, .95])
    STATS[c] = {"dtype": DTYPES[c],
                "p5": qs.iloc[0], "p25": qs.iloc[1], "p50": qs.iloc[2],
                "p75": qs.iloc[3], "p95": qs.iloc[4],
                "min": s.min(), "max": s.max(),
                "zero_rate": round(float((s == 0).mean()), 3),
                "empty_rate": round(float(s.isna().mean()), 3)}

IDS = {"flow_key": "timeframe_id|proto|ip_a|ip_b|port_a|port_b",
       "ip_a_format": "anon_ip_N",
       "ip_a_unique": int(df.ip_a.nunique()),
       "ip_b": PINNED["ip_b"],
       "port_a_top": df.port_a.value_counts().head(5).to_dict(),
       "port_b_range": [int(df.port_b.min()), int(df.port_b.max())],
       "timeframe_range": [int(df.timeframe_id.min()), int(df.timeframe_id.max())],
       "proto_values": df.proto.value_counts().to_dict()}

print("pinned (capture):", len(PINNED), list(PINNED))
print("constant in benign but LLM-free:", FREE_CONST)
print("constraints:", len(CONSTRAINTS))
print("target_ip_pkts_total median now:", STATS["target_ip_pkts_total"]["p50"])

In [ ]:
RULES = {
    "source": "benign.csv", "n_rows": len(df),
    "column_order": df.columns.tolist(), "dtypes": DTYPES,
    "core_columns": CORE,
    "derived": DERIVED,
    "flow_key_rule": IDS["flow_key"],
    "pinned": PINNED,
    "always_empty": ALL_NA,
    "empty_rules": EMPTY_RULES,
    "constraints": CONSTRAINTS,
    "benign_stats": STATS,
    "id_formats": IDS,
}
path = OUT / "rules.json"
path.write_text(json.dumps(RULES, indent=1, default=str))
print("saved", path, path.stat().st_size, "bytes")
print(f"{df.shape[1]} cols = {len(CORE)} core (LLM) + {len(DERIVED)} derived"
      f" + 1 flow_key + {len(PINNED)} pinned + {len(ALL_NA)} always-empty"
      f" + {len(LABEL)} label")

from google.colab import files
files.download(str(path))

In [ ]:
R = json.loads((OUT / "rules.json").read_text())
ORDER, DT, CORE = R["column_order"], R["dtypes"], R["core_columns"]
DER, PIN, ALLNA, ST = R["derived"], R["pinned"], R["always_empty"], R["benign_stats"]
IDS = ["timeframe_id", "proto", "ip_a", "ip_b", "port_a", "port_b"]

def num(v):
    if v is None: return "NA"
    v = float(v)
    if v == 0: return "0"
    if abs(v) >= 1000: return f"{v:.0f}"
    if abs(v) < 1e-3: return f"{v:.3g}"
    return f"{v:.4g}"

def kind(c):
    t = DT[c]
    frac = re.search(r"(ratio|fraction|share)", c)
    if t == "int" and (frac or (c in DER and "/" in DER[c])): return "decimal"
    return {"int": "whole number", "float": "decimal", "text": "text"}[t]

def readable(e):
    e = e.replace("1.0 * ", "")
    m = re.fullmatch(r"\(\((\w+) > 0\) & \((\w+) == 0\)\)", e)
    if m: return f"1 if {m[1]} > 0 and {m[2]} is 0, otherwise 0"
    m = re.fullmatch(r"\((\w+) (==|>) 0\)", e)
    if m: return f"1 if {m[1]} {'is 0' if m[2] == '==' else '> 0'}, otherwise 0"
    return e

MEANING = {
 "port_a": "source port the attacker sends from",
 "port_b": "port on the monitored host being hit",
 "pkts_fwd": "packets the source sent in this flow",
 "bytes_fwd": "bytes the source sent in this flow",
 "duration": "seconds from first to last packet (0 if only one packet)",
 "pkt_size_std": "spread of packet sizes (bytes)",
 "pkt_size_min": "smallest packet (bytes)",
 "pkt_size_max": "largest packet (bytes)",
 "mean_hop_count": "INT hop count (1 here when present)",
 "mean_hop_latency": "INT average hop latency",
 "latency_min": "INT lowest hop latency",
 "latency_max": "INT highest hop latency",
 "latency_std": "INT spread of hop latency",
 "mean_iat_int_ns": "average time between packets (ns)",
 "iat_std_int_ns": "spread of time between packets (ns)",
 "iat_min_int_ns": "shortest time between packets (ns)",
 "iat_max_int_ns": "longest time between packets (ns)",
 "syn_count_fwd": "packets with SYN", "ack_count_fwd": "packets with ACK",
 "fin_count_fwd": "packets with FIN", "rst_count_fwd": "packets with RST",
 "psh_count_fwd": "packets with PSH", "urg_count_fwd": "packets with URG",
 "small_pkt_fraction_total": "share of small packets (0-1)",
 "tiny_pkt_fraction_total": "share of tiny packets (0-1)",
 "queue_occupancy_mean": "INT average switch queue occupancy",
 "queue_occupancy_std": "INT spread of queue occupancy",
 "queue_occupancy_max": "INT highest queue occupancy",
 "truncated_udp_fraction": "share of truncated UDP packets (0-1)",
 "target_service_flow_count": "flows hitting this host+port this timeframe (all sources)",
 "target_service_unique_peer_ips": "different sources hitting this host+port",
 "target_service_unique_peer_ports": "different source ports hitting this host+port",
 "target_service_pkts_total": "packets to this host+port this timeframe (all sources)",
 "target_service_bytes_total": "bytes to this host+port this timeframe (all sources)",
 "target_ip_flow_count": "flows to the monitored host this timeframe (all sources)",
 "target_ip_unique_peer_ips": "different sources sending to the host this timeframe",
 "target_ip_unique_peer_ports": "different source ports sending to the host",
 "target_ip_unique_services": "different ports hit on the host",
 "target_ip_pkts_total": "packets to the host this timeframe (all sources)",
 "target_ip_bytes_total": "bytes to the host this timeframe (all sources)",
 "source_flow_count": "flows this source sent this timeframe",
 "source_unique_target_ips": "different hosts this source sent to",
 "source_unique_target_ports": "different ports this source hit",
 "source_unique_peer_ports": "different source ports this source used",
 "source_pkts_total": "packets this source sent this timeframe (all its flows)",
 "source_bytes_total": "bytes this source sent this timeframe (all its flows)",
}

FIXED_NOW = {"timeframe_id": "the timeframe given at the end", "proto": "6",
             "ip_a": "anon_ip_21", "is_attack": "True"}
pins = [c for c in PIN if c not in IDS]
gen = IDS + pins + [c for c in CORE if c not in IDS]
tail = list(DER) + ["flow_key"] + ALLNA + ["is_attack"]
GEN_ORDER = gen + [c for c in tail if c not in gen]
assert sorted(GEN_ORDER) == sorted(ORDER)

spec = []
for i, c in enumerate(GEN_ORDER, 1):
    if c in FIXED_NOW: how = f"always {FIXED_NOW[c]}"
    elif c in PIN: how = f"always {PIN[c] if isinstance(PIN[c], str) else num(PIN[c])}"
    elif c in ALLNA: how = "always NA"
    elif c == "flow_key": how = "= timeframe_id|proto|ip_a|ip_b|port_a|port_b"
    elif c in DER:
        how = "= " + readable(DER[c])
        if "/ duration" in DER[c]: how += "  (NA if duration is 0)"
    else: how = "you choose: " + MEANING.get(c, "realistic value")
    spec.append(f"{i}. {c} ({kind(c)}) - {how}")

prof = []
for c in CORE:
    if c not in ST or c in IDS: continue
    s = ST[c]
    note = []
    if s["zero_rate"] > 0: note.append(f"0 in {s['zero_rate']:.0%}")
    if s["empty_rate"] > 0: note.append(f"NA in {s['empty_rate']:.0%}")
    prof.append(f"- {c}: typical {num(s['p25'])} to {num(s['p75'])}, "
                f"middle {num(s['p50'])}, range {num(s['p5'])} to {num(s['p95'])}"
                f" (max {num(s['max'])})" + (f"; {', '.join(note)}" if note else ""))

rules_extra = [c for c in R["constraints"] if not re.fullmatch(r"\w+ >= 0", c)]
rules_extra += ["pkts_total <= target_service_pkts_total <= target_ip_pkts_total",
                "pkts_total <= source_pkts_total <= target_ip_pkts_total",
                "target_service_flow_count <= target_ip_flow_count"]

PROMPT = f"""You are helping us simulate network ATTACK traffic for a real research network. We will use your records to train an intrusion detection system, so they need to be realistic and follow this network's rules exactly. You only generate attack traffic, never normal traffic.

1. THE NETWORK
AmLight is an international research-and-education network run by FIU/CIARA, linking the United States, Latin America, the Caribbean, and South Africa (12 sites, 24+ long-haul 100-Gbps links, 4.9 Tbps total).
Traffic is recorded at a monitoring point with INT (In-band Network Telemetry), which adds hop count, hop latency, and switch queue occupancy to each flow. We only see traffic coming IN to one monitored host, anon_ip_2. Replies from that host are never seen, so every flow has zero backward packets.
The recording is split into 13 timeframes (1 to 13). Each record is one flow in one timeframe, identified by timeframe, protocol, source IP (ip_a), destination IP (ip_b), source port (port_a), and destination port (port_b).

2. WHAT NORMAL TRAFFIC LOOKS LIKE HERE (learn the environment from this, do not copy it)
These numbers come from real benign traffic in this network. Use them to learn the scales, units, and formats of each field. Normal traffic here comes from about {R['id_formats']['ip_a_unique']} different hosts, mostly HTTPS servers (source port 443) sending to high ports on anon_ip_2. Most normal flows are tiny: over half are a single packet with duration 0. The target_* numbers were measured while no attack was running.
{chr(10).join(prof)}

3. THE ATTACK TO SIMULATE
A single outside machine, anon_ip_21, runs a controlled denial-of-service (DoS) attack against anon_ip_2 during timeframes 5 to 11. Timeframes 5-10 are the full attack; timeframe 11 is where it winds down. All attack traffic comes from this one source, which can send many flows to several ports on the target.
The attack is a TCP ACK/PSH flood: the attacker sends a high, steady stream of TCP packets with the ACK flag set, some also with PSH, without ever doing a TCP handshake. The target has to check every packet against a connection that does not exist, which wears it down. Flood packets usually carry large payloads to also eat bandwidth. SYN, FIN, RST, and URG play no part in this attack.
Think about how such a flood would look in this network compared with the normal traffic above, and choose values accordingly.

4. THE 113 FIELDS, IN THE ORDER YOU WRITE THEM
"you choose" = pick a realistic value for this attack.
"=" = calculate it exactly from fields earlier in the same record.
"always" = write exactly that value.
{chr(10).join(spec)}

5. RULES FOR EMPTY VALUES
- Write NA for empty values, never leave a field blank.
- duration is 0 exactly when pkts_fwd is 1, and above 0 when pkts_fwd is 2 or more.
- pps_* and bps_* fields are NA exactly when duration is 0.
- The 4 time-between-packets fields (mean_iat_int_ns, iat_std_int_ns, iat_min_int_ns, iat_max_int_ns) are all NA when pkts_fwd is 1 or when INT data is missing; they may be NA when pkts_fwd is 2; they must be numbers when pkts_fwd is 3 or more.
- The 8 INT fields (mean_hop_count, mean_hop_latency, latency_min, latency_max, latency_std, queue_occupancy_mean, queue_occupancy_std, queue_occupancy_max) are either all filled or all NA; they may be NA only when pkts_fwd is 2 or less. When filled, mean_hop_count is 1.

6. RULES TO KEEP RECORDS REALISTIC
- No negative numbers.
{chr(10).join('- ' + c for c in rules_extra)}

7. OUTPUT FORMAT
- Reply with one JSON object: {{"records": [ ... ]}}, holding exactly the number of records requested at the end.
- Each record is an object with all 113 fields of section 4 as keys.
- Numbers as JSON numbers. ip_a, ip_b, flow_key and is_attack as strings (is_attack is "True").
- Wherever this prompt says NA, write null.
- Each record must be different and fit its phase of the attack.

NOW GENERATE: {{K}} attack records for timeframe {{TIMEFRAME}} ({{PHASE}}).
"""

(OUT / "prompt_attack.txt").write_text(PROMPT)
(OUT / "generation_order.json").write_text(json.dumps(GEN_ORDER))
print(f"prompt: {len(PROMPT)} chars (~{len(PROMPT)//4} tokens)")
print(f"fields: {len(GEN_ORDER)} | you-choose: {sum('you choose' in s for s in spec)}"
      f" | formulas: {sum(' = ' in s for s in spec)} | always: {sum('always' in s for s in spec)}")
print(PROMPT[:600])

from google.colab import files
files.download(str(OUT / "prompt_attack.txt"))
files.download(str(OUT / "generation_order.json"))

In [ ]:
import json, re
from pathlib import Path

OUT = Path("/content/rules_out")
OUT.mkdir(exist_ok=True, parents=True)
print("rules.json present:", (OUT / "rules.json").exists())